# CSIRO Biomass Prediction - ResNet50 Model
Kaggle submission notebook using ResNet50 transfer learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

## 1. Load Data

In [ ]:
# Load data from local path
dataset_path = Path('csiro-biomass')
train_df = pd.read_csv(dataset_path / 'train.csv')
test_df = pd.read_csv(dataset_path / 'test.csv')
sample_submission = pd.read_csv(dataset_path / 'sample_submission.csv')

print(f"Training set: {train_df.shape[0]} rows, {train_df.shape[1]} columns") 
print(f"Test set: {test_df.shape[0]} rows, {test_df.shape[1]} columns")
print(f"Sample submission: {sample_submission.shape[0]} rows")
print(f"\nTarget columns: {train_df['target_name'].unique()}")

## 2. Data Preprocessing

In [ ]:
# Configuration
IMAGE_SIZE = 224
BATCH_SIZE = 16

# Dataset statistics (pre-calculated from training images)
dataset_mean = [0.4379, 0.5048, 0.3002]
dataset_variance = [0.0633, 0.0626, 0.0576]

# Target column names
target_columns = train_df['target_name'].unique()
print(f"Targets: {list(target_columns)}")

In [ ]:
# Pivot data so each image has one row with all 5 targets
train_pivoted = train_df.pivot_table(
    index=['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'],
    columns='target_name',
    values='target'
).reset_index()

print(f"Pivoted data shape: {train_pivoted.shape}")
print(f"Unique images: {train_pivoted['image_path'].nunique()}")

# Train/Validation split (stratified by State)
train_data, val_data = train_test_split(
    train_pivoted, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_pivoted['State']
)

print(f"Train: {len(train_data)} images, Val: {len(val_data)} images")

In [ ]:
# Image augmentation and preprocessing pipelines
train_augmentation = keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(factor=15/360),
    layers.RandomBrightness(factor=0.2),
    layers.RandomContrast(factor=0.2),
    layers.Rescaling(1./255),
    layers.Normalization(mean=dataset_mean, variance=dataset_variance)
])

val_preprocessing = keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.Rescaling(1./255),
    layers.Normalization(mean=dataset_mean, variance=dataset_variance)
])

# Build preprocessing layers with sample image
sample_path = dataset_path / train_df['image_path'].iloc[0]
sample_img = tf.io.read_file(str(sample_path))
sample_img = tf.image.decode_jpeg(sample_img, channels=3)
sample_img = tf.cast(sample_img, tf.float32)
_ = train_augmentation(tf.expand_dims(sample_img, 0))
_ = val_preprocessing(tf.expand_dims(sample_img, 0))
print("Preprocessing pipelines built successfully")

In [ ]:
# Dataset creation functions
def create_dataset(df, preprocessing_fn, batch_size=32, shuffle=True):
    """Create a tf.data.Dataset from dataframe with image paths and targets."""
    image_paths = [str(dataset_path / p) for p in df['image_path'].values]
    targets = df[target_columns].values.astype('float32')
    
    def load_and_preprocess(path, target):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.cast(img, tf.float32)
        img = tf.expand_dims(img, 0)
        img = preprocessing_fn(img)
        img = tf.squeeze(img, 0)
        return img, target
    
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, targets))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df))
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

def create_test_dataset(image_paths, preprocessing_fn, batch_size=32):
    """Create test dataset for inference."""
    full_paths = [str(dataset_path / p) for p in image_paths]
    
    def load_and_preprocess(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.cast(img, tf.float32)
        img = tf.expand_dims(img, 0)
        img = preprocessing_fn(img)
        img = tf.squeeze(img, 0)
        return img
    
    dataset = tf.data.Dataset.from_tensor_slices(full_paths)
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Create datasets
train_dataset = create_dataset(train_data, train_augmentation, batch_size=BATCH_SIZE, shuffle=True)
val_dataset = create_dataset(val_data, val_preprocessing, batch_size=BATCH_SIZE, shuffle=False)

# Verify
for images, targets in train_dataset.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Target batch shape: {targets.shape}")

## 3. Build Model

In [ ]:
def build_baseline_model(num_targets=5):
    """Build baseline model with ResNet50 backbone."""
    
    # Load pre-trained ResNet50
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        pooling='avg'
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build model
    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_targets, activation='linear')(x)
    
    model = keras.Model(inputs, outputs)
    return model

# Build and compile model
model = build_baseline_model(num_targets=len(target_columns))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

model.summary()

## 4. Train Model

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'resnet50_model_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

print("Training with frozen backbone...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Load saved model
model = keras.models.load_model('resnet50_model_best.keras')
#model = keras.models.load_model('baseline_model_best.keras')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training vs Validation MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest validation loss: {min(history.history['val_loss']):.4f}")
print(f"Best validation MAE: {min(history.history['val_mae']):.4f}")

In [ ]:
# Validation predictions
val_predictions = model.predict(val_dataset)
val_targets = np.concatenate([y for x, y in val_dataset], axis=0)
# Calculate overall MAE for validation set
overall_mae = np.mean(np.abs(val_predictions - val_targets))
print(f"Overall Validation MAE: {overall_mae:.4f}")
print(val_predictions[:5])
print(val_targets[:5])
for images, targets in val_dataset.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Target batch shape: {targets.shape}")


In [ ]:
# Display validation images with predictions
def show_predictions_with_images(val_data, val_predictions, target_columns, dataset_path, num_images=6):
    """Display sample validation images with actual and predicted values."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    # Get random indices
    indices = np.random.choice(len(val_data), num_images, replace=False)
    
    for i, idx in enumerate(indices):
        # Load image
        img_path = dataset_path / val_data.iloc[idx]['image_path']
        img = Image.open(img_path)
        
        # Get actual and predicted values
        actual = val_data.iloc[idx][target_columns].values.astype(float)
        predicted = val_predictions[idx]
        
        # Display image
        axes[i].imshow(img)
        axes[i].axis('off')
        
        # Create title with actual vs predicted
        title_lines = []
        for j, col in enumerate(target_columns):
            title_lines.append(f"{col[:8]}: {actual[j]:.2f} → {predicted[j]:.2f}")
        
        axes[i].set_title('\n'.join(title_lines), fontsize=8, ha='center')
    
    plt.suptitle('Validation Images: Actual → Predicted', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Show sample predictions
show_predictions_with_images(val_data.reset_index(drop=True), val_predictions, target_columns, dataset_path, num_images=6)

## 5. Generate Kaggle Submission

In [ ]:
# Get unique test images
test_unique = test_df.drop_duplicates(subset='image_path')[['image_path']].reset_index(drop=True)
print(f"Unique test images: {len(test_unique)}")

# Create test dataset
test_dataset = create_test_dataset(
    test_unique['image_path'].values, 
    val_preprocessing, 
    batch_size=BATCH_SIZE
)

# Make predictions
print("Making predictions on test set...")
predictions = model.predict(test_dataset, verbose=1)
print(f"Predictions shape: {predictions.shape}")

In [ ]:
# Create submission dataframe
print(f"Target columns order: {list(target_columns)}")

# Create predictions dataframe with image paths
pred_df = pd.DataFrame(predictions, columns=target_columns)
pred_df['image_path'] = test_unique['image_path'].values

# Match predictions to test_df sample_id
submission_rows = []
for _, row in test_df.iterrows():
    image_path = row['image_path']
    target_name = row['target_name']
    sample_id = row['sample_id']
    
    pred_row = pred_df[pred_df['image_path'] == image_path]
    if len(pred_row) > 0:
        pred_value = pred_row[target_name].values[0]
        # Ensure non-negative predictions
        pred_value = max(0, pred_value)
        submission_rows.append({
            'sample_id': sample_id,
            'target': pred_value
        })

submission_df = pd.DataFrame(submission_rows)
print(f"Submission shape: {submission_df.shape}")
submission_df.head(10)

In [ ]:
# Save submission file
submission_filename = 'resnet50_submission.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"Submission saved to: {submission_filename}")

# Validate format
print(f"\nSample submission rows: {len(sample_submission)}")
print(f"Our submission rows: {len(submission_df)}")

## 6. Advanced Models + Comparison

Advanced techniques used:
- Transfer learning with stronger backbones
- Two-stage fine-tuning (freeze then partial unfreeze)
- AdamW optimizer + Huber loss
- Stronger regularization (BatchNorm + Dropout)
- Validation-based learning-rate reduction and early stopping

In [ ]:
from tensorflow.keras.applications import DenseNet121, EfficientNetV2B0

# Fallback baseline metrics if history is unavailable in current kernel session
baseline_best_val_loss = 337.9422
baseline_best_val_mae = 12.0844
if 'history' in globals() and hasattr(history, 'history') and 'val_loss' in history.history and 'val_mae' in history.history:
    baseline_best_val_loss = float(min(history.history['val_loss']))
    baseline_best_val_mae = float(min(history.history['val_mae']))

def build_transfer_model(backbone_name, num_targets=5):
    if backbone_name == 'densenet121':
        backbone = DenseNet121(
            include_top=False,
            weights='imagenet',
            input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
            pooling='avg'
        )
    elif backbone_name == 'efficientnetv2b0':
        backbone = EfficientNetV2B0(
            include_top=False,
            weights='imagenet',
            input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
            pooling='avg'
        )
    else:
        raise ValueError('Unsupported backbone')

    backbone.trainable = False

    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = backbone(inputs, training=False)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(num_targets, activation='linear')(x)

    model = keras.Model(inputs, outputs, name=f'{backbone_name}_advanced')
    return model, backbone

def compile_advanced_model(model, lr=3e-4):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-5),
        loss=keras.losses.Huber(delta=5.0),
        metrics=['mae']
    )

def train_two_stage(model, backbone, train_ds, val_ds, stage1_epochs=8, stage2_epochs=8):
    callbacks_stage1 = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    ]

    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=stage1_epochs,
        callbacks=callbacks_stage1,
        verbose=1
    )

    backbone.trainable = True
    unfreeze_from = int(len(backbone.layers) * 0.7)
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= unfreeze_from

    compile_advanced_model(model, lr=2e-5)

    callbacks_stage2 = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
    ]

    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=stage2_epochs,
        callbacks=callbacks_stage2,
        verbose=1
    )

    combined_history = {}
    for key in history1.history.keys():
        combined_history[key] = history1.history[key] + history2.history.get(key, [])
    return combined_history

def evaluate_on_validation(model, val_ds):
    val_loss, val_mae = model.evaluate(val_ds, verbose=0)
    y_true = np.concatenate([y for _, y in val_ds], axis=0)
    y_pred = model.predict(val_ds, verbose=0)
    val_rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return float(val_loss), float(val_mae), val_rmse

In [ ]:
advanced_results = []

advanced_model_configs = [
    {
        'name': 'DenseNet121 Improved',
        'backbone_key': 'densenet121',
        'model_file': 'densenet121_improved_model.keras',
        'submission_file': 'densenet121_improved_submission.csv',
        'techniques': 'DenseNet121 transfer learning + two-stage fine-tuning + AdamW + Huber loss'
    },
    {
        'name': 'EfficientNetV2B0 Improved',
        'backbone_key': 'efficientnetv2b0',
        'model_file': 'efficientnetv2b0_improved_model.keras',
        'submission_file': 'efficientnetv2b0_improved_submission.csv',
        'techniques': 'EfficientNetV2B0 transfer learning + two-stage fine-tuning + AdamW + Huber loss'
    }
]

for cfg in advanced_model_configs:
    print(f"\nTraining {cfg['name']}...")
    adv_model, adv_backbone = build_transfer_model(cfg['backbone_key'], num_targets=len(target_columns))
    compile_advanced_model(adv_model, lr=3e-4)

    _ = train_two_stage(
        adv_model,
        adv_backbone,
        train_dataset,
        val_dataset,
        stage1_epochs=8,
        stage2_epochs=8
    )

    val_loss, val_mae, val_rmse = evaluate_on_validation(adv_model, val_dataset)
    print(f"{cfg['name']} - val_loss: {val_loss:.4f}, val_mae: {val_mae:.4f}, val_rmse: {val_rmse:.4f}")

    adv_model.save(cfg['model_file'])
    print(f"Saved model: {cfg['model_file']}")

    # Submission generation
    test_unique = test_df.drop_duplicates(subset='image_path')[['image_path']].reset_index(drop=True)
    test_dataset_adv = create_test_dataset(
        test_unique['image_path'].values,
        val_preprocessing,
        batch_size=BATCH_SIZE
    )
    test_predictions = adv_model.predict(test_dataset_adv, verbose=1)

    pred_df = pd.DataFrame(test_predictions, columns=target_columns)
    pred_df['image_path'] = test_unique['image_path'].values

    submission_rows = []
    for _, row in test_df.iterrows():
        image_path = row['image_path']
        target_name = row['target_name']
        sample_id = row['sample_id']

        pred_row = pred_df[pred_df['image_path'] == image_path]
        if len(pred_row) > 0:
            pred_value = pred_row[target_name].values[0]
            pred_value = max(0, pred_value)
            submission_rows.append({'sample_id': sample_id, 'target': pred_value})

    submission_adv = pd.DataFrame(submission_rows)
    submission_adv.to_csv(cfg['submission_file'], index=False)
    print(f"Saved submission: {cfg['submission_file']} | shape={submission_adv.shape}")

    advanced_results.append({
        'model': cfg['name'],
        'advanced_techniques': cfg['techniques'],
        'val_loss': val_loss,
        'val_mae': val_mae,
        'val_rmse': val_rmse,
        'submission_file': cfg['submission_file']
    })

In [ ]:
comparison_rows = [
    {
        'model': 'Baseline ResNet50',
        'advanced_techniques': 'Transfer learning + augmentation',
        'val_loss': baseline_best_val_loss,
        'val_mae': baseline_best_val_mae,
        'val_rmse': np.nan,
        'submission_file': 'resnet50_submission.csv'
    }
]

comparison_rows.extend(advanced_results)
comparison_df = pd.DataFrame(comparison_rows).sort_values('val_mae', ascending=True).reset_index(drop=True)

comparison_df.to_csv('model_comparison_results.csv', index=False)
comparison_df.to_markdown('model_comparison_results.md', index=False)

print('Model comparison table:')
display(comparison_df)

print('\nSaved comparison files:')
print('- model_comparison_results.csv')
print('- model_comparison_results.md')

## 7. Comparison Table From Saved Models
Use saved model files to compute validation metrics and create the final comparison table.

Notes:
- `baseline_model_best.keras` is treated as the EfficientNet baseline.
- `resnet50_model_best.keras` is treated as the ResNet50 baseline.

In [ ]:
from pathlib import Path

saved_model_specs = [
    {
        'model': 'Baseline EfficientNet (baseline_model_best)',
        'file': 'baseline_model_best.keras',
        'advanced_techniques': 'EfficientNet transfer learning baseline'
    },
    {
        'model': 'Baseline ResNet50',
        'file': 'resnet50_model_best.keras',
        'advanced_techniques': 'ResNet50 transfer learning baseline'
    },
    {
        'model': 'DenseNet121 Improved',
        'file': 'densenet121_improved_model.keras',
        'advanced_techniques': 'DenseNet121 + two-stage fine-tuning + AdamW + Huber loss'
    },
    {
        'model': 'EfficientNetV2B0 Improved',
        'file': 'efficientnetv2b0_improved_model.keras',
        'advanced_techniques': 'EfficientNetV2B0 + two-stage fine-tuning + AdamW + Huber loss'
    }
]

val_targets_saved_eval = np.concatenate([y for _, y in val_dataset], axis=0)
saved_model_rows = []

for spec in saved_model_specs:
    model_path = Path(spec['file'])
    if not model_path.exists():
        print(f"Skipping {spec['model']} (missing file: {spec['file']})")
        continue

    loaded_model = keras.models.load_model(model_path)
    eval_loss, eval_mae = loaded_model.evaluate(val_dataset, verbose=0)
    preds = loaded_model.predict(val_dataset, verbose=0)
    eval_rmse = float(np.sqrt(np.mean((val_targets_saved_eval - preds) ** 2)))

    saved_model_rows.append({
        'model': spec['model'],
        'advanced_techniques': spec['advanced_techniques'],
        'val_loss': float(eval_loss),
        'val_mae': float(eval_mae),
        'val_rmse': eval_rmse,
        'model_file': spec['file']
    })

saved_models_comparison_df = pd.DataFrame(saved_model_rows).sort_values('val_mae', ascending=True).reset_index(drop=True)

saved_models_comparison_df.to_csv('model_comparison_from_saved_models.csv', index=False)
saved_models_comparison_df.to_markdown('model_comparison_from_saved_models.md', index=False)

print('Comparison from saved models:')
display(saved_models_comparison_df)

print('\nSaved files:')
print('- model_comparison_from_saved_models.csv')
print('- model_comparison_from_saved_models.md')

In [ ]:
# Efficientnet model
efficientnet = keras.models.load_model("models/baseline_model_best.keras")

In [ ]:
# Load DenseNet model from results and inspect layers (Keras compatibility-safe)
from pathlib import Path

densenet_path = Path('results') / 'densenet121_improved_model.keras'

# Some saved .keras files include `quantization_config` in layer configs.
# Older Keras builds may fail on that field, so patch Dense only while loading.
_original_dense_init = keras.layers.Dense.__init__
def _dense_init_compat(self, *args, **kwargs):
    kwargs.pop('quantization_config', None)
    return _original_dense_init(self, *args, **kwargs)

keras.layers.Dense.__init__ = _dense_init_compat
try:
    densenet = keras.models.load_model(densenet_path, compile=False)
finally:
    keras.layers.Dense.__init__ = _original_dense_init

print(f"Loaded: {densenet_path}")
densenet.summary()

# Compact layer table for quick inspection
layer_table = pd.DataFrame([
    {
        'index': i,
        'name': layer.name,
        'type': type(layer).__name__,
        'trainable': layer.trainable,
        'params': int(layer.count_params())
    }
    for i, layer in enumerate(densenet.layers)
])
display(layer_table)

## Parameters and Layers of Models

In [ ]:
from collections import Counter
from pathlib import Path

model_files = {
    'Baseline EfficientNet': 'baseline_model_best.keras',
    'Baseline ResNet50': 'resnet50_model_best.keras',
    'DenseNet121 Improved': 'densenet121_improved_model.keras',
    'EfficientNetV2B0 Improved': 'efficientnetv2b0_improved_model.keras',
}

def get_backbone_layer(m):
    for lyr in m.layers:
        if isinstance(lyr, keras.Model):
            lname = lyr.name.lower()
            if any(k in lname for k in ['resnet50', 'efficientnet', 'densenet121']):
                return lyr
    return None

def get_backbone_name(m):
    b = get_backbone_layer(m)
    if b is None:
        return 'Unknown'
    lname = b.name.lower()
    if 'resnet50' in lname:
        return 'ResNet50'
    if 'efficientnetv2-b0' in lname or 'efficientnetv2b0' in lname:
        return 'EfficientNetV2B0'
    if 'efficientnet' in lname:
        return 'EfficientNet'
    if 'densenet121' in lname:
        return 'DenseNet121'
    return b.name

rows = []
for model_name, filename in model_files.items():
    p = Path(filename)
    if not p.exists():
        print(f'Skipping {model_name}: missing {filename}')
        continue
    m = keras.models.load_model(p)
    backbone = get_backbone_layer(m)

    layer_types = Counter(type(l).__name__ for l in m.layers)
    top_types = ', '.join([f'{k}:{v}' for k, v in layer_types.most_common(6)])

    backbone_layers = len(backbone.layers) if backbone is not None else 0
    if backbone is not None:
        bb_types = Counter(type(l).__name__ for l in backbone.layers)
        bb_top_types = ', '.join([f'{k}:{v}' for k, v in bb_types.most_common(8)])
    else:
        bb_top_types = ''

    rows.append({
        'model': model_name,
        'file': filename,
        'backbone': get_backbone_name(m),
        'n_layers_total': len(m.layers),
        'n_backbone_layers': backbone_layers,
        'trainable_layers': sum(1 for l in m.layers if l.trainable),
        'params_total': int(m.count_params()),
        'params_trainable': int(sum(np.prod(w.shape) for w in m.trainable_weights)),
        'params_non_trainable': int(sum(np.prod(w.shape) for w in m.non_trainable_weights)),
        'head_layer_types': top_types,
        'backbone_layer_types': bb_top_types
    })

layer_diff_df = pd.DataFrame(rows).sort_values('params_total', ascending=False).reset_index(drop=True)
display(layer_diff_df)

In [ ]:
layer_diff_df['backbone_layer_types']